# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, focused on the FAIR² dataset package: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Overview of record sets in the dataset
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for i, record_set in enumerate(record_sets):
    print(f"Record Set {i+1} | @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', 'N/A')}")
    print(f"  Description: {record_set.get('description', 'N/A')}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    - @id: {field['@id']} | name: {field.get('name','[No Name]')}")
    print()

## 3. Data Extraction
Load data from specific record set(s) into DataFrame(s) for analysis. Record set and field `@id`s are used for reference.

Let's extract all record sets to explore further.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for {record_set_id} with shape: {df.shape}")

# Example: Print columns for the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nColumns in {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

For demonstration, let's select a numeric field from the first record set.

In [ ]:
# Select record set and a numeric field for analysis
import numpy as np

record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# You may need to adjust the field name below based on the dataset
# By convention, mass spectrometry protein tables often have fields like 'Molecular_Weight' or 'MW', use the correct column
# We'll pick the first numeric-like column available

numeric_field = None
for col in df.columns:
    try:
        # Try converting to numeric to check if it's a numeric field
        pd.to_numeric(df[col], errors='raise')
        numeric_field = col
        break
    except:
        continue

if numeric_field is None:
    print("No numeric fields found in the DataFrame.")
else:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filter out records where the numeric field is greater than a threshold
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a plausible group field (e.g., 'Sample', or the first string column)
    group_field = None
    for col in df.columns:
        if (df[col].dtype == object or df[col].dtype.name == 'category') and col != numeric_field:
            group_field = col
            break

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot the distribution of the numeric field and its normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and not filtered_df.empty:
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, color='orange')
    plt.title(f"Normalized {numeric_field} (filtered)")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        # Boxplot by group
        plt.figure(figsize=(10,3))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading dataset metadata and records using Croissant's `@id` referencing system.
- Record sets, their fields, and data were explored programmatically.
- Simple EDA and visualization provided insight into the numeric distribution and categorical groupings.

You are encouraged to continue exploring specific fields and analyses relevant to your research!